<a href="https://colab.research.google.com/github/CristianCarrereAlvarez/monitor-mercado-laboral/blob/main/SMLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Monitor Mercado Laboral Chile

Código en el repo, datos en Drive. Ver `CLAUDE.md` para el contexto.

## Qué corre acá y qué no

| etapa | dónde |
|---|---|
| **captura** | **solo en tu máquina** — Colab está bloqueado, ver abajo |
| consolidación | tu máquina (recomendado) o acá |
| análisis | acá, que ya trae pandas |

Este notebook es, sobre todo, **la herramienta de análisis**. Corré la
sección 1 al abrir la sesión y andá a la 3.


---

## 1. Preparación

Las dos celdas, en orden. Colab corta por inactividad (~90 min), así que
hay que repetirlas al reabrir. **No hace falta instalar nada**: pandas ya
viene, y Playwright no se usa acá.


### 1.1 Drive — define `DATOS`

Todo lo demás depende de esta variable. Si salta un `NameError`, es que
esta celda no se corrió.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATOS = '/content/drive/MyDrive/monitor_mercado_laboral'
os.makedirs(f'{DATOS}/crudo', exist_ok=True)
os.makedirs(f'{DATOS}/maestras', exist_ok=True)

print('crudo   :', sorted(os.listdir(f'{DATOS}/crudo')))
print('maestras:', sorted(os.listdir(f'{DATOS}/maestras')))

### 1.2 Repositorio

Necesario para `consolidar.py` y para el catálogo de carreras.


In [ ]:
import os, subprocess

REPO = 'https://github.com/CristianCarrereAlvarez/monitor-mercado-laboral.git'

if not os.path.exists('/content/repo'):
    !git clone $REPO /content/repo
else:
    !git -C /content/repo pull --ff-only

print(subprocess.run(['git','-C','/content/repo','log','--oneline','-1'],
                     capture_output=True, text=True).stdout)
print(sorted(os.listdir('/content/repo')))

**Si `git pull` falla** con *"untracked working tree file would be overwritten"*,
re-cloná limpio con esta celda (es seguro: en el repo no hay datos):

In [ ]:
import shutil
shutil.rmtree('/content/repo', ignore_errors=True)
!git clone $REPO /content/repo
!ls /content/repo

---

## 2. La captura no corre acá

**Akamai bloquea los rangos de datacenter.** Desde Colab,
`trabajando.cl` devuelve 403 en todo, incluida la portada. No se arregla
con headers ni esperando: lo único que mira es la IP de origen.

Se corre desde una máquina con IP residencial, en la terminal:

```bash
cd ~/monitor-mercado-laboral && git pull

./mensual.sh                  # las 10 áreas + consolidación
./capturar.sh "Agropecuaria"  # una sola área
./mensual.sh --desde "Salud"  # retomar sin repetir
```

Los envoltorios resuelven la carpeta de datos solos y eligen el modo de
sesión. No hay que hacer `cd` a ningún lado.

El crudo cae en Drive y sube solo, así que las celdas de abajo lo
encuentran sin que hagas nada.


---

## 3. Consolidación

Lee todos los `crudo_*.jsonl` y hace upsert sobre las ocho maestras. Es
**idempotente**. Las columnas manuales que agregues a mano
(`carrera_sies`, `rubro`, `tamano`…) **sobreviven** a cada corrida.

`mensual.sh` ya consolida al terminar. Esta celda sirve si capturaste
suelto con `capturar.sh`, o si cambiaste un criterio de parseo y querés
reprocesar sin volver a scrapear.

⚠️ Si acabás de capturar, esperá a que Drive termine de subir el crudo.


In [ ]:
%cd /content/repo
!git pull --ff-only
%cd $DATOS
!python /content/repo/consolidar.py --crudo crudo --maestras maestras

---

## 4. Análisis

Sobre las maestras consolidadas. Es para lo que este notebook sirve hoy.


### 4.1 Distribución de carreras declaradas

Debería ser bimodal: valores chicos y después un salto a los avisos genéricos.
`UMBRAL_AVISO_GENERICO = 30` corta en el hueco.

In [ ]:
import pandas as pd
a = pd.read_csv(f'{DATOS}/maestras/avisos.csv')
print(a.n_carreras_declaradas.value_counts().sort_index().to_string())

### 4.2 Los avisos genéricos

Un puñado de empleadores declara casi todo el catálogo. Aparecen en la búsqueda
de cualquier carrera y hay que excluirlos de cualquier análisis por carrera.

In [ ]:
print(a[a.n_carreras_declaradas > 30]
      [['aviso_id','empresa_id','titulo','comuna','n_carreras_declaradas','n_instituciones']]
      .sort_values('n_carreras_declaradas', ascending=False)
      .to_string(index=False, max_colwidth=45))

### 4.3 Concentración por empleador

**Leer antes de cualquier agregado por área.** En Derecho un solo empleador
aportó el 35% de los avisos con su boilerplate institucional. El conteo de
avisos mide publicación, no demanda.

In [ ]:
top = a.empresa_id.value_counts()
n = len(a)
for k in (1, 3, 10):
    print(f'top {k:2d} empleadores: {top.head(k).sum()*100/n:.1f}% de {n} avisos')
print()
print(a.groupby('empresa_id').size().sort_values(ascending=False).head(10).to_string())

### 4.4 Ruta A — término de búsqueda → SIES

**Sobre-atribuye por diseño.** El término es lo que se buscó, no lo que el aviso
declara: en Derecho los 342 avisos quedan etiquetados «Derecho», incluidos los
que no son jurídicos. Es señal de contexto, no clasificación.

`n_terminos_sin_mapeo > 0` significa que el crudo tiene términos que ya no están
en el catálogo — detector de deriva.

In [ ]:
at = pd.read_csv(f'{DATOS}/maestras/aviso_termino.csv')
print(at.groupby(['termino_busqueda','carrera_sies','areas_sies']).size()
        .sort_values(ascending=False).head(20).to_string())
print()
print('avisos con términos sin mapeo:', int((a.n_terminos_sin_mapeo > 0).sum()))

### 4.5 Panel longitudinal — duración de vacante

**No mezclar las tres calidades.** Solo `observada` es una medición real;
`cota_superior` es un techo y `censurada` sigue viva. Hacen falta al menos dos
corridas del mismo aviso para tener una sola medición.

In [ ]:
print(a.calidad_duracion.value_counts().to_string())
print()
obs = a[a.calidad_duracion == 'observada']['dias_publicado_hasta_baja'].dropna()
print(obs.describe().to_string() if len(obs) else
      'sin duraciones observadas todavía — hace falta una segunda corrida')

### 4.6 Prioridad de homologación

Ordenado por `n_avisos_especificos`, que ignora los avisos genéricos. Es el orden
correcto para completar a mano `carrera_sies` en `carreras_trabajando.csv`.

In [ ]:
ct = pd.read_csv(f'{DATOS}/maestras/carreras_trabajando.csv')
print(ct.head(30)[['carrera_trabajando','n_avisos_especificos','n_avisos_acum']]
        .to_string(index=False))